# JEM - Coupling Quick Start

This notebook demonstrates a JAX-ESM (JEM) example using JAX-GCM (JCM), Slab Ocean Model, and Slab Land Model.

## Import Packages

In [ ]:
import os, sys
from pathlib import Path

# or `export PYTHONPATH=/path/to/jax-esm/root/directory`
sys.path.append( (Path(os.getcwd()) / ".." ).resolve())

In [ ]:
import jcm
from jcm.geometry import Geometry
import jax_datetime as jdt

from jem.tool_scripts.generate_jcm_forcing_and_topography_files import (
    generate_jcm_forcing_and_topography_files,
)
from jem.components import JCM, SlabLandModel, SlabOceanModel
from jem.mapping import IdentityRegridder
from jem.mapping import BasicForcingMapper
from jem.base.coupler import Coupler
import jem.utils.tree_tools as tree_tools

## Configurations

In [ ]:
start_datetime = jdt.to_datetime("2000-01-01")
coupling_timestep = jdt.to_timedelta(1, "day")
simulation_interval = jdt.to_timedelta(10, "day")
output_dir = Path("output/JCM_SOM_SLM").resolve()

external_files = generate_jcm_forcing_and_topography_files()
output_dir.mkdir(exist_ok=True, parents=True)
geometry = Geometry.from_file(external_files["terrain"])
one_second = jdt.to_timedelta(1, "second")

## Create Components

In [ ]:
# Creating components

atm_model = jcm.model.Model(
    start_date=start_datetime,
    geometry=geometry
)

JCM.make_jem_compatible(
    atm_model,
    coupling_timestep=coupling_timestep,
    save_interval=jdt.to_timedelta(12, "hour"),
)

components = dict(
    atm=atm_model,
    ocn=SlabOceanModel(
        start_datetime=start_datetime,
        mask_file=external_files["terrain"],
        SST_clim_file=external_files["forcing"],
    ),
    lnd=SlabLandModel(
        start_datetime=start_datetime,
        topography_file=external_files["terrain"],
        mask_file=external_files["terrain"],
        land_clim_file=external_files["forcing"],
    ),
)

## Creating Flux and Scalar Exchange between Components

In [ ]:
# Creating regridders and mapping
identity_regridder = IdentityRegridder()
forcing_mapper = BasicForcingMapper(components=components)
forcing_mapper.add_forcing_mapping(
    source = ("atm", "extra.total_heat_flux"),
    target = ("ocn", "flux.total_heat_flux"),
    regridder = identity_regridder,
)
forcing_mapper.add_forcing_mapping(
    source = ("ocn", "prog.sea_surface_temperature"),
    target = ("atm", "sea_surface_temperature"),
    regridder = identity_regridder,
)
forcing_mapper.add_forcing_mapping(
    source = ("atm", "extra.total_heat_flux"),
    target = ("lnd", "flux.total_heat_flux"),
    regridder = identity_regridder,
)
forcing_mapper.add_forcing_mapping(
    source = ("lnd", "prog.land_surface_temperature"),
    target = ("atm", "stl_am"),
    regridder = identity_regridder,
)

## Create Coupled Model

In [ ]:
model = Coupler(
    components=components,
    forcing_mappers=dict(fm=forcing_mapper),
)

print("Model info: ") 
tree_tools.print_tree(model.get_info(), root="Model")

## Run Coupled Model

In [ ]:
# Obtain initial condition
initial_coupled_state_forcing = model.initialize()

print(initial_coupled_state_forcing["ocn"][0]["prog.mixed_layer_depth"])

print("Model state:")
tree_tools.print_tree(initial_coupled_state_forcing, root="ModelState")

print("Create model trajectory function...")
trajectory_function = model.generate_trajectory_function(
    workflow=["fm", "atm", "ocn", "lnd"],
    iterations = int(simulation_interval / coupling_timestep),
)

# Run coupled model
print("Running model...")
state_holder, predictions = trajectory_function(initial_coupled_state_forcing)

## Output into NetCDF

In [ ]:
output_dict = model.predictions_to_xarray(predictions)
for component_name, ds in output_dict.items():
    output_file = output_dir / f"{component_name:s}.nc"
    print("Output file: ", str(output_file))
    ds.to_netcdf(output_file, engine="netcdf4")

## Visualization

In [ ]:
import matplotlib.pyplot as plt

### Atmosphere

In [ ]:
ds = output_dict["atm"]
print(str(ds))

#### Precipitation

In [ ]:
ds['condensation.precls'].plot(x='lon', y='lat', col='time', col_wrap=2, aspect=2)
ds['convection.precnv'].plot(x='lon', y='lat', col='time', col_wrap=2, aspect=2)

#### Moisture

In [ ]:
ds['specific_humidity'].mean('lon').plot(x='lat', y='level', col='time', col_wrap=3, aspect=6, yincrease=False)
ds['specific_humidity'].isel(level=3).plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)

#### Clouds

In [ ]:
ds['shortwave_rad.cloudc'].plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)
ds['shortwave_rad.qcloud'].plot(x='lon', y='lat', col='time', col_wrap=3, aspect=2)

### Ocean

In [ ]:
ds = output_dict["ocn"]
print(str(ds))

#### Sea Surface Temperature
We plot both the SST and its difference with respect to the initial condition.

In [ ]:
g = (ds['sea_surface_temperature'] - 273.15).plot(x='longitude', y='latitude', col='time', col_wrap=3, aspect=2, cmap="gnuplot")
g.fig.suptitle("Sea Surface Temperature [${}^\\circ \\mathrm{C}$]", fontsize=16)

In [ ]:
g = (ds['sea_surface_temperature'] - ds['sea_surface_temperature'].isel(time=0)).plot(x='longitude', y='latitude', col='time', col_wrap=3, aspect=2, center=0)
g.fig.suptitle("Difference of Sea Surface Temperature between labeled time and initial condition [${}^\\circ \\mathrm{C}$]", fontsize=16)

#### Total heat flux

In [ ]:
g = ds['total_heat_flux'].plot(x='longitude', y='latitude', col='time', col_wrap=3, aspect=2, cmap="bwr_r", center=0.0)
g.fig.suptitle("Total heat flux (upward positive) [$\\mathrm{W} / \\mathrm{m}^2$]", fontsize=16)

### Land

In [ ]:
ds = output_dict["lnd"]
print(str(ds))

#### Land Surface Temperature

In [ ]:
g = (ds['land_surface_temperature'] - 273.15).plot(x='longitude', y='latitude', col='time', col_wrap=3, aspect=2, cmap="bwr", center=0)
g.fig.suptitle("Land Surface Temperature [${}^\\circ \\mathrm{C}$]", fontsize=16, y=1.02)

In [ ]:
plt.show()